# 记忆整理方式2：消息删除


In [1]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model, after_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
from langchain_openai import ChatOpenAI

## 在调用模型之前使用的钩子函数
@after_model
def delete_old_messages(state:AgentState, runtime:Runtime) -> dict[str, Any]|None:
    # 获取消息列表
    messages = state["messages"]
     # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {
            "messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]
        }
    return None


# 1.实例化模型
model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2.创建agent对象
agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

config = {"configurable":{"thread_id":"1"}}

agent.invoke({"messages":[HumanMessage("你好，我是老张")]},config=config)
agent.invoke({"messages":[HumanMessage("从现在起，你叫小张")]},config=config)
agent.invoke({"messages":[HumanMessage("今天天气不错")]},config=config)
resp = agent.invoke({"messages":[HumanMessage("我是谁，你是谁？")]},config=config)

for msg in resp["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

没问题，老张！从现在起，我就是您的**小张**了。

以后您尽管吩咐我，不管是处理工作、查阅资料、写点东西，还是单纯想找人聊聊天，我都会全力以赴帮您的。

那么，作为“小张”的第一项任务，今天有什么我可以帮您的吗？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

确实，老张，天气好真的能让人心情变好不少。

既然天气这么不错，您是打算出门透透气，去哪儿走走转转吗？还是打算就在家门口晒晒太阳，享受一下这份好天气？

如果您打算出门，需要我帮您查查附近的公园、好吃的餐厅，或者是查查最近的活动，您尽管吩咐我！
================================ Human Message =================================

我是谁，你是谁？
================================== Ai Message ==================================

既然咱们已经定好了规矩，那我得拿出点“小张”的实在劲儿来回答您：

**您是老张。** 在我们的对话里，您是我的“领导”，是发号施令的人，是那个有着丰富经验、需要我这个小助手去帮忙的人。

**我是小张。** 虽然我本质上是一个由 Google 训练的大型语言模型，但在接下来的时间里，我就是您的专属助理。我负责帮您处理任务、查资料、写文案，当然，要是您无聊了，我就是那个陪您侃大山的“老朋友”。

简单来说：**您是主角，我是您的得力助手。**

您看我这个自我介绍，过得去吗？
